In [1]:
##import the libraries and load the data set
import pandas as pd
import numpy as np


df = pd.read_csv("Auto Sales data.csv")
df.head()

,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,DAYS_SINCE_LASTORDER,STATUS,PRODUCTLINE,MSRP,PRODUCTCODE,CUSTOMERNAME,PHONE,ADDRESSLINE1,CITY,POSTALCODE,COUNTRY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30,95.70,2,2871.00,24/02/2018,828,Shipped,Motorcycles,95,S10_1678,Land of Toys Inc.,2125557818,897 Long Airport Avenue,NYC,10022,USA,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,07/05/2018,757,Shipped,Motorcycles,95,S10_1678,Reims Collectables,26.47.1555,59 rue de l'Abbaye,Reims,51100,France,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,01/07/2018,703,Shipped,Motorcycles,95,S10_1678,Lyon Souveniers,+33 1 46 62 7555,27 rue du Colonel Pierre Avia,Paris,75508,France,Da Cunha,Daniel,Medium
3,10145,45,83.26,6,3746.70,25/08/2018,649,Shipped,Motorcycles,95,S10_1678,Toys4GrownUps.com,6265557265,78934 Hillside Dr.,Pasadena,90003,USA,Young,Julie,Medium
4,10168,36,96.66,1,3479.76,28/10/2018,586,Shipped,Motorcycles,95,S10_1678,Technics Stores Inc.,6505556809,9408 Furth Circle,Burlingame,94217,USA,Hirano,Juri,Medium


In [2]:
##make a copy of the dataset
clean_df = df.copy()

In [7]:
##changing the column names to lower case
clean_df.columns = (
    clean_df.columns.str.strip().str.lower().str.replace(" ","_"))

clean_df.columns

Index(['ordernumber', 'quantityordered', 'priceeach', 'orderlinenumber',
       'sales', 'orderdate', 'days_since_lastorder', 'status', 'productline',
       'msrp', 'productcode', 'customername', 'phone', 'addressline1', 'city',
       'postalcode', 'country', 'contactlastname', 'contactfirstname',
       'dealsize'],
      dtype='str')

In [8]:
text_columns = clean_df.select_dtypes(
    include="object"
).columns

text_columns

<ipython-input-8-df1c62751632>:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = clean_df.select_dtypes(


Index(['orderdate', 'status', 'productline', 'productcode', 'customername',
       'phone', 'addressline1', 'city', 'postalcode', 'country',
       'contactlastname', 'contactfirstname', 'dealsize'],
      dtype='str')

In [9]:
for column in text_columns:
    clean_df[column] = clean_df[column].str.strip()

In [10]:
##clean the date
clean_df["orderdate"] = pd.to_datetime(clean_df["orderdate"], dayfirst = True)

In [14]:
##check numerical columns
numerical_column = ["ordernumber","quantityordered","priceeach","orderlinenumber","sales","msrp"]
clean_df[numerical_column].dtypes

ordernumber          int64
quantityordered      int64
priceeach          float64
orderlinenumber      int64
sales              float64
msrp                 int64
dtype: object

In [15]:
##check for duplicates
clean_df.duplicated().sum()

np.int64(0)

In [16]:
##check missing values
clean_df.isnull().sum()

ordernumber             0
quantityordered         0
priceeach               0
orderlinenumber         0
sales                   0
orderdate               0
days_since_lastorder    0
status                  0
productline             0
msrp                    0
productcode             0
customername            0
phone                   0
addressline1            0
city                    0
postalcode              0
country                 0
contactlastname         0
contactfirstname        0
dealsize                0
dtype: int64

In [17]:
clean_df["quantityordered"].describe()

count    2747.000000
mean       35.103021
std         9.762135
min         6.000000
25%        27.000000
50%        35.000000
75%        43.000000
max        97.000000
Name: quantityordered, dtype: float64

In [20]:
##analytical date column
##helps us analyze sales by year, month, quarter
clean_df["year"] = clean_df["orderdate"].dt.year
clean_df["month"] = clean_df["orderdate"].dt.month
clean_df["month_name"] = clean_df["orderdate"].dt.month_name()
clean_df["quarter"] = clean_df["orderdate"].dt.quarter



In [22]:
##creating a year month field
clean_df["year_month"] = (clean_df["orderdate"].dt.to_period("M").astype(str))
clean_df["year_month"]

0       2018-02
1       2018-05
2       2018-07
3       2018-08
4       2018-10
         ...   
2742    2019-12
2743    2020-01
2744    2020-03
2745    2020-03
2746    2020-05
Name: year_month, Length: 2747, dtype: str

In [27]:
##calculate the discount using the manufacturer suggested retail price(msrp)
clean_df["discount_amount"] = (clean_df["msrp"] - clean_df["priceeach"])

##percentage discount
clean_df["percent_discount"] = ((clean_df["discount_amount"]/clean_df["msrp"]) * 100).round(2)

##check the discount calculations
clean_df[
    [
        "msrp",
        "priceeach",
        "discount_amount",
        "percent_discount"
    ]
    ]

,msrp,priceeach,discount_amount,percent_discount
0,95,95.70,-0.70,-0.74
1,95,81.35,13.65,14.37
2,95,94.74,0.26,0.27
3,95,83.26,11.74,12.36
4,95,96.66,-1.66,-1.75
...,...,...,...,...
2742,54,112.22,-58.22,-107.81
2743,54,137.19,-83.19,-154.06
2744,54,125.99,-71.99,-133.31
2745,54,62.24,-8.24,-15.26


In [29]:
clean_df["percent_discount"].describe()

count    2747.000000
mean       -4.038828
std        38.099609
min      -666.270000
25%       -12.160000
50%        -0.400000
75%        11.520000
max        77.130000
Name: percent_discount, dtype: float64

In [30]:
print("Rows:", clean_df.shape[0])
print("Columns:", clean_df.shape[1])

print("\nMissing values:")
print(clean_df.isnull().sum().sum())

print("\nDuplicate rows:")
print(clean_df.duplicated().sum())

print("\nDate range:")
print(clean_df["orderdate"].min(), "to", clean_df["orderdate"].max())

Rows: 2747
Columns: 27

Missing values:
0

Duplicate rows:
0

Date range:
2018-01-06 00:00:00 to 2020-05-31 00:00:00


In [31]:
clean_df.to_csv("auto sales cleaned.csv", index = False)